# 10. 위험도 등급: 추정 밝기 + 2단계 로지스틱 (주 모델) vs 2단계 규칙

**왜 만들었나 (11/6)**: 07은 "밝기를 먼저 맞히고 → 등급으로 바꾸는" 구조라 작은 밝기 오차에도 등급이 뒤집혀 입구 F1 .526에 머묾.
모델②처럼 **등급을 2단계로 바로 맞히는** 구조로 바꿔 비교 (EXP-91). 11/6 형 결정: 주 모델 = 2단계 로지스틱. 04:12 형 결정: **사진으로 추정한 바깥 밝기(직선 회귀)를 2단계의 변수로 넣음** → 사진 → 바깥 밝기 → 위험도 흐름 유지, 같은 밝기를 동공 모델에 넘김.

| 항목 | 내용 |
|---|---|
| 시험 종류 | **②a**: 71786만으로 학습 → 처음 보는 카메라(Mapillary)로 시험. 08 도로 사진(②b)은 안 씀 |
| 학습 | `data/aihub_71786/학습표_71786.csv` (07이 저장) 3,624장 = 2023년 8~9월 입구 1,927 + 터널 없는 고속도로 1,697. 겨울 0장 |
| 시험 | `data/시험표_mapillary.csv` (07과 같은 사진 숫자, 11/6 저장) 118장 중 **입구 66장 (51장면)만** 씀. 출구는 뺌 (11/5 형 결정: 암순응 = 입구만 봄. 71786 학습에도 출구 0장) |
| 정답 | 사진마다 천리안 일사량 → L20 → 휘도비 → 하(<10)·중(10~15)·상(>15). 07과 같은 공식 |
| 입력 | 사진 숫자 10개 + 해 높이 + 구름 + **추정 바깥 밝기** (직선 회귀, 같은 12개 숫자로 추정. 학습 사진은 5겹 OOF 추정값만 씀 = 자기 정답 안 봄). **달은 뺌** (학습이 8~9월뿐) |
| 튜닝 | 71786 안에서만 장면 단위 5겹 OOF Macro F1로 고름. Mapillary는 고를 때 안 봄 |
| 채점 | 입구 66장 3등급 Macro F1 + 장면 1,000번 다시 뽑기 95% 범위. 우위 = 짝지은 차이 95% 하한 > 0 (모델②와 같음) |

실행 위치: `sunsight_video` 폴더. 07을 한 번 끝까지 돌려 둔 상태여야 함 (학습표 · 시험표 필요).

In [ ]:
import json, numpy as np, pandas as pd, warnings; warnings.filterwarnings('ignore')
from pathlib import Path
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import f1_score, accuracy_score
DATA = Path('data')
# 07의 공통 함수(정답 공식 · 등급 · 물리 공식 · 손 규칙)를 그대로 가져옴 → 07과 같은 자
nb7 = json.load(open('07_밝기추정_규칙vsML.ipynb', encoding='utf-8'))
src5 = [''.join(c['source']) for c in nb7['cells'] if 'class Rule1Hand' in ''.join(c['source'])][0]
exec(src5.replace('import lightgbm as lgb\n', 'pass\n'))
FEATURES = ['sky_mean', 'blue_sky', 'white_ratio', 'shadow_contrast', 'road_mean', 'sky_road_ratio', 'portal_ratio', 'saturation', 'spread', 'mean_all']
PLUS = FEATURES + ['sun_alt', 'cloud']
LAB = ['하', '중', '상']
def gr(y, cls): return grade(ratio_from_L20(10 ** np.asarray(y), np.asarray(cls)))
f1 = lambda a, b: f1_score(a, b, average='macro', labels=LAB, zero_division=0)

tr = pd.read_csv(DATA / 'aihub_71786' / '학습표_71786.csv')
mp = pd.read_csv(DATA / '시험표_mapillary.csv', dtype={'image_id': str}, parse_dates=['captured_at'])
test = mp[mp.test_ok & (mp.cls == '입구')].reset_index(drop=True); ent = np.ones(len(test), bool)   # 입구만 (출구 뺌)
gtr, gte = gr(tr.y, tr.cls), gr(test.y, test.cls); ok = tr.test_ok.values.astype(bool)
print('학습 71786', len(tr), '장 | 달', tr.month.value_counts().sort_index().to_dict(), '| 입구 정답', pd.Series(gtr[ok]).value_counts().to_dict())
print('시험 Mapillary 입구', len(test), '장 ·', test[ent].scene.nunique(), '장면 | 입구 정답', pd.Series(gte[ent]).value_counts().to_dict())
print('시험 입구 달별 장수', test[ent].month.value_counts().sort_index().to_dict())

## 1. 모델 정의
- **주 모델 = 추정 밝기 + 2단계 로지스틱**: ① 직선 회귀(Ridge, 사진 10개 + 해 높이 + 구름)로 바깥 밝기 log L20 추정 → ② 그 추정 밝기를 13번째 변수로 넣어 2단계 로지스틱
- **2단계 로지스틱 (밝기 없이, 11/6 오전 주 모델)**: 1단계 "하 vs 위험(중·상)" → 위험이면 2단계 "중 vs 상". 각 단계 로지스틱 회귀 (빈칸=학습 중앙값, 표준화, 등급 수 가중)
- **2단계 규칙**: 같은 2단계인데 단계마다 숫자 하나 + 경계 하나(if 한 줄). 숫자와 경계는 학습 사진에서 고름
- 비교: 07 주 모델(부스팅 회귀 → 등급, 07 튜닝 값), 규칙4 물리 공식, 규칙1 손 규칙

In [ ]:
X = lambda d, c: d[c].to_numpy(float)
folds = list(GroupKFold(5).split(tr, groups=tr.scene.values))
def two_stage(make, cols):
    """1단계: 하인가 위험인가 / 2단계: (위험만) 중인가 상인가"""
    def fit_pred(a, b, return_models=False):
        ga = gr(a.y, a.cls); s2 = ga != '하'
        m1 = make().fit(X(a, cols), ga != '하'); m2 = make().fit(X(a, cols)[s2], ga[s2] == '상')
        r1 = m1.predict(X(b, cols)).astype(bool); r2 = m2.predict(X(b, cols)).astype(bool)
        out = np.where(~r1, '하', np.where(r2, '상', '중'))
        return (out, m1, m2) if return_models else out
    return fit_pred
def oof(fp):        # 71786 안에서 장면 단위 5겹 → 입구 사진 Macro F1 (튜닝은 이 숫자로만)
    out = np.empty(len(tr), dtype=object)
    for a, b in folds: out[b] = fp(tr.iloc[a], tr.iloc[b])
    return f1(gtr[ok], out[ok])
logit = lambda C: (lambda: make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(C=C, class_weight='balanced', max_iter=2000)))
stump = lambda: make_pipeline(SimpleImputer(strategy='median'), DecisionTreeClassifier(max_depth=1, class_weight='balanced', random_state=0))

tune = {}
for nm, cols in {'사진+해높이+구름': PLUS, '해높이+구름': ['sun_alt', 'cloud'], '해높이만': ['sun_alt'], '사진만': FEATURES}.items():
    sc = {C: round(oof(two_stage(logit(C), cols)), 3) for C in [0.01, 0.1, 1, 10]}
    tune[nm] = (max(sc, key=sc.get), sc); print('2단계 로지스틱', nm, '학습 OOF F1', sc, '→ C =', tune[nm][0])

In [ ]:
PRED = {}
for nm, cols in {'사진+해높이+구름': PLUS, '해높이+구름': ['sun_alt', 'cloud'], '해높이만': ['sun_alt'], '사진만': FEATURES}.items():
    PRED[f'2단계 로지스틱 ({nm})'] = two_stage(logit(tune[nm][0]), cols)(tr, test)
PRED['2단계 규칙 (if 한 줄씩)'] = two_stage(stump, PLUS)(tr, test)
hgb = HistGradientBoostingRegressor(learning_rate=0.1, max_leaf_nodes=31, max_iter=100, min_samples_leaf=10, l2_regularization=0.0, random_state=0)
PRED['07 주 모델 (부스팅 회귀→등급)'] = gr(hgb.fit(X(tr, PLUS), tr.y).predict(X(test, PLUS)), test.cls)
PRED['규칙4 물리 공식'] = gr(Rule4Physics().predict(test), test.cls)
PRED['규칙1 손 규칙'] = gr(Rule1Hand().fit(tr, tr.y).predict(test), test.cls)
print('학습 OOF F1 · 2단계 규칙', round(oof(two_stage(stump, PLUS)), 3))

### 주 모델: 추정 밝기 + 2단계 로지스틱
- 밝기 추정 = 직선 회귀(Ridge α=10). 학습 사진의 추정 밝기는 학습 안에서 다시 5겹으로 나눠 "자기 정답을 안 본" 값만 씀 (새는 길 막기)
- 시험 사진의 추정 밝기 = 학습 전체로 학습한 직선 회귀
- 규제 C는 {0.01, 0.1, 1} 중 학습 OOF F1로 고름

In [ ]:
from sklearn.linear_model import Ridge
ridge = lambda: make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge(alpha=10))
def add_est(a, b):
    """a로 직선 회귀 학습 → b 밝기 추정. a 자신의 추정 밝기는 a 안 5겹 OOF"""
    ea = np.empty(len(a))
    for x, y in GroupKFold(5).split(a, groups=a.scene.values):
        ea[y] = ridge().fit(a.iloc[x][PLUS], a.y.iloc[x]).predict(a.iloc[y][PLUS])
    a = a.copy(); b = b.copy(); a['est_L'] = ea; b['est_L'] = ridge().fit(a[PLUS], a.y).predict(b[PLUS]); return a, b
MAIN_COLS = PLUS + ['est_L']
def main_fp(C):
    def fp(a, b, return_models=False):
        a2, b2 = add_est(a, b); return two_stage(logit(C), MAIN_COLS)(a2, b2, return_models)
    return fp
sc = {C: round(oof(main_fp(C)), 3) for C in [0.01, 0.1, 1]}; C_MAIN = max(sc, key=sc.get)
print('주 모델 학습 OOF F1', sc, '→ C =', C_MAIN)
main = '주 모델: 추정 밝기 + 2단계 로지스틱'
trE, teE = add_est(tr, test)
PRED[main] = main_fp(C_MAIN)(tr, test)
from sklearn.metrics import r2_score
print('추정 밝기 R² (log L20) · 입구', round(r2_score(test.y, teE.est_L), 3))

## 2. 채점: 입구 66장

In [ ]:
sce = test.scene.values[ent]; ug = np.unique(sce); idx = {g: np.where(sce == g)[0] for g in ug}
rng = np.random.default_rng(0); BS = [np.concatenate([idx[g] for g in rng.choice(ug, len(ug))]) for _ in range(1000)]
ge = gte[ent]; B = {}; rows = []
for k, p in PRED.items():
    p = np.asarray(p); pe = p[ent]; B[k] = np.array([f1(ge[i], pe[i]) for i in BS])
    rec = lambda c: round(float(((pe == c) & (ge == c)).sum() / max((ge == c).sum(), 1)), 2)
    rows.append({'모델': k, '입구 F1': round(f1(ge, pe), 3), '95% 범위': f'{np.percentile(B[k], 2.5):.2f}~{np.percentile(B[k], 97.5):.2f}',
                 '하 재현': rec('하'), '중 재현': rec('중'), '상 재현': rec('상')})
res = pd.DataFrame(rows).sort_values('입구 F1', ascending=False)
res.to_csv(DATA / '등급2단계_결과.csv', index=False, encoding='utf-8-sig'); res

In [ ]:
pairs = [(main, '2단계 로지스틱 (사진+해높이+구름)'), (main, '2단계 규칙 (if 한 줄씩)'), (main, '규칙4 물리 공식'), (main, '규칙1 손 규칙'), (main, '07 주 모델 (부스팅 회귀→등급)'),
         ('2단계 규칙 (if 한 줄씩)', '규칙4 물리 공식'),
         # 카메라(사진)를 쓰는 값어치: 같은 2단계 구조에서 사진 피처만 빼고 비교
         ('2단계 로지스틱 (사진+해높이+구름)', '2단계 로지스틱 (해높이+구름)'), (main, '2단계 로지스틱 (해높이+구름)'),
         ('2단계 로지스틱 (사진+해높이+구름)', '2단계 로지스틱 (해높이만)'), (main, '2단계 로지스틱 (해높이만)')]   # 사진 없는 쪽 최고(.759)와도 비교
pr = pd.DataFrame([{'A': a, 'B': b, 'F1 차이': round(float((B[a] - B[b]).mean()), 3), '95% 하한': round(float(np.percentile(B[a] - B[b], 2.5)), 3),
                    '95% 상한': round(float(np.percentile(B[a] - B[b], 97.5)), 3)} for a, b in pairs])
pr['A 우위 (하한>0)'] = pr['95% 하한'] > 0
pr.to_csv(DATA / '등급2단계_짝차이.csv', index=False, encoding='utf-8-sig'); pr

## 3. 약점 점검: 계절로 갈리는지
시험 입구 66장 중 12월(정답 전부 하)·5월(정답 전부 상)이 50장. 나머지 16장(중 9장 포함)이 진짜 어려운 부분

In [ ]:
rows = []
for k in PRED:
    p = np.asarray(PRED[k]); r = {'모델': k}
    for nm, m in [('12월', test.month == 12), ('5월', test.month == 5), ('나머지 달', ~test.month.isin([5, 12]))]:
        mm = m.values & ent; r[f'{nm} {mm.sum()}장 정확도'] = round(accuracy_score(gte[mm], p[mm]), 2)
    rows.append(r)
season = pd.DataFrame(rows); season.to_csv(DATA / '등급2단계_달별.csv', index=False, encoding='utf-8-sig'); season

## 4. 주 모델 · 규칙이 무엇을 보나 (가중치 · 경계)

In [ ]:
_, m1, m2 = main_fp(C_MAIN)(tr, test, return_models=True)
coef = pd.DataFrame({'1단계 (위험인가) 가중치': m1[-1].coef_[0], '2단계 (상인가) 가중치': m2[-1].coef_[0]}, index=MAIN_COLS).round(2)
coef = coef.reindex(coef['1단계 (위험인가) 가중치'].abs().sort_values(ascending=False).index)
coef.to_csv(DATA / '등급2단계_가중치.csv', encoding='utf-8-sig'); display(coef)
_, s1, s2 = two_stage(stump, PLUS)(tr, test, return_models=True)
for nm, s in [('1단계', s1), ('2단계', s2)]:
    t = s[-1].tree_; print(f'2단계 규칙 {nm}: {PLUS[t.feature[0]]} <= {t.threshold[0]:.1f} 이면', '하' if nm == '1단계' else '중', '/ 넘으면', '위험(2단계로)' if nm == '1단계' else '상')

## 5. 사진별 결과 저장 (동공 모델 · 09에서 씀)

In [ ]:
out = test[['image_id', 'cls', 'captured_at', 'month', 'sun_alt', 'cloud']].copy(); out['정답 등급'] = gte
for k in [main, '2단계 로지스틱 (사진+해높이+구름)', '2단계 규칙 (if 한 줄씩)', '07 주 모델 (부스팅 회귀→등급)', '규칙4 물리 공식']: out[k] = PRED[k]
out['정답 L20'] = (10 ** test.y.values).round(1)
out['추정 L20 (주 모델)'] = (10 ** teE.est_L.values).round(1)   # 동공 모델에 넘길 바깥 밝기
out['추정 휘도비 (주 모델)'] = ratio_from_L20(10 ** teE.est_L.values, test.cls.values).round(2)
out['추정밝기 등급 (참고)'] = gr(teE.est_L.values, test.cls.values)
print('주 모델 등급 vs 추정 밝기로 낸 등급 일치율 (입구)', round(float((out[main].values[ent] == out['추정밝기 등급 (참고)'].values[ent]).mean()), 2))
out.to_csv(DATA / '등급2단계_사진별.csv', index=False, encoding='utf-8-sig'); print('저장: data/등급2단계_*.csv'); out.head()

## 6. 구조 교차 비교 (모델②와 같은 방식, 11/6 형 요청)
- 목적: "ML이 규칙보다 낫다"를 구조 탓이 아니라 학습 탓인지 확인. ML에서 잘 된 구조(추정 밝기 + 2단계)를 규칙에도, 규칙에서 잘 된 구조(해 높이 하나로 2단계 경계)를 ML에도 씌워 같은 자로 비교
- 데이터: 학습 71786 3,624장(8~9월, 입구 1,927 + 터널 없는 고속도로 1,697), 시험 Mapillary 입구 66장 · 51장면(②a). 규칙5 · 출구 뺌
- 고르는 법: 구조마다 규칙 쪽 · ML 쪽 후보 중 **학습 71786 장면 5겹 OOF F1 1등**만 시험에 냄 (시험 점수 보고 고르지 않음)
- 채점: 입구 66장 Macro F1, 장면 1,000번 bootstrap, 짝지은 차이 95% 하한 > 0이면 우위
- 보조: 학습 쪽 OOF(입구 1,927장 · 736장면)에서도 같은 짝지은 차이. 같은 카메라라 "새 카메라 일반화" 근거는 아님

In [ ]:
# ===== 구조 교차 비교: ML에서 잘 된 구조 → 규칙에, 규칙에서 잘 된 구조 → ML에 (모델②와 같은 방식) =====
# 데이터·채점은 위와 같음: 학습 71786 (장면 5겹 OOF로만 고름), 시험 Mapillary 입구 66장 (②a), 장면 bootstrap 1,000번
RULE_L = {'규칙1 손 규칙': Rule1Hand, '규칙2 경계값 학습': Rule2Learned, '규칙3 숫자 하나 직선': Rule3OneLine, '규칙4 물리 공식': Rule4Physics}
def add_rule_L(R):
    def f(a, b):     # 규칙 밝기를 13번째 변수로. 학습 쪽은 학습 안 5겹 OOF (주 모델과 같은 방식)
        ea = np.empty(len(a))
        for x, y in GroupKFold(5).split(a, groups=a.scene.values):
            ea[y] = R().fit(a.iloc[x], a.y.iloc[x].values).predict(a.iloc[y])
        a = a.copy(); b = b.copy(); a['est_L'] = ea; b['est_L'] = R().fit(a, a.y.values).predict(b); return a, b
    return f
def with_L(adder, make, cols):
    def fp(a, b):
        a2, b2 = adder(a, b); return two_stage(make, cols)(a2, b2)
    return fp
def reg_grade(R):    # 구조 ① 밝기 추정 → 공식 등급
    return lambda a, b: gr(R().fit(a, a.y.values).predict(b), b.cls.values)
class RidgeReg:
    def fit(self, X, y): self.m = ridge().fit(X[PLUS], y); return self
    def predict(self, X): return self.m.predict(X[PLUS])

CAND = {}   # 구조 → {'규칙': {이름: fp}, 'ML': {이름: fp}}
CAND['① 밝기 추정 → 공식 등급'] = {'규칙': {k: reg_grade(R) for k, R in RULE_L.items()},
                                   'ML': {'직선 회귀 → 등급': reg_grade(RidgeReg)}}
CAND['② 2단계 직접 분류 (사진+해높이+구름)'] = {'규칙': {'2단계 if 한 줄씩': two_stage(stump, PLUS)},
                                   'ML': {f'2단계 로지스틱 C={C}': two_stage(logit(C), PLUS) for C in [0.01, 0.1, 1]}}
CAND['③ 추정 밝기 + 2단계 (ML 최고 구조 → 규칙에)'] = {'규칙': {f'{k} 밝기 + 2단계 if': with_L(add_rule_L(R), stump, PLUS + ['est_L']) for k, R in RULE_L.items()},
                                   'ML': {f'직선 회귀 밝기 + 2단계 로지스틱 C={C}': main_fp(C) for C in [0.01, 0.1, 1]}}
CAND['④ 해 높이 하나로 2단계 (규칙 최고 구조 → ML에)'] = {'규칙': {'2단계 if (해 높이만)': two_stage(stump, ['sun_alt'])},
                                   'ML': {f'2단계 로지스틱 (해 높이만) C={C}': two_stage(logit(C), ['sun_alt']) for C in [0.01, 0.1, 1, 10]}}
XR, XB = [], {}
for st, d in CAND.items():
    for side in ['규칙', 'ML']:
        sc = {k: oof(fp) for k, fp in d[side].items()}; best = max(sc, key=sc.get)   # 학습 OOF로만 고름
        p = np.asarray(d[side][best](tr, test)); key = f'{st} | {side}'
        XB[key] = np.array([f1(ge[i], p[ent][i]) for i in BS])
        XR.append({'구조': st, '쪽': side, '고른 것 (학습 OOF 1등)': best, '학습 OOF F1': round(sc[best], 3),
                   '입구 F1': round(f1(ge, p[ent]), 3), '95% 범위': f'{np.percentile(XB[key], 2.5):.2f}~{np.percentile(XB[key], 97.5):.2f}'})
        print(key, '→', best, '| OOF', round(sc[best], 3), '| 입구 F1', XR[-1]['입구 F1'])
xr = pd.DataFrame(XR); xr.to_csv(DATA / '등급2단계_구조교차.csv', index=False, encoding='utf-8-sig')
# 같은 구조 안에서 ML − 규칙, 그리고 ML 최고 구조(③ ML) − 각 규칙
best_ml = '③ 추정 밝기 + 2단계 (ML 최고 구조 → 규칙에) | ML'
XP = []
for st in CAND:
    XP.append((f'{st} | ML', f'{st} | 규칙'))
for st in CAND:
    if f'{st} | 규칙' != best_ml: XP.append((best_ml, f'{st} | 규칙'))
xp = pd.DataFrame([{'A': a, 'B': b, 'F1 차이': round(float((XB[a] - XB[b]).mean()), 3), '95% 하한': round(float(np.percentile(XB[a] - XB[b], 2.5)), 3),
                    '95% 상한': round(float(np.percentile(XB[a] - XB[b], 97.5)), 3)} for a, b in XP])
xp['A 우위 (하한>0)'] = xp['95% 하한'] > 0
xp.to_csv(DATA / '등급2단계_구조교차_짝차이.csv', index=False, encoding='utf-8-sig')
display(xr); display(xp)


In [ ]:
# ===== 보조: 학습 쪽 71786 안 장면 5겹 OOF에서도 짝지은 차이 (같은 카메라 · 표본이 커서 범위가 좁음) =====
def oof_pred(fp):
    out = np.empty(len(tr), dtype=object)
    for a, b in folds: out[b] = fp(tr.iloc[a], tr.iloc[b])
    return out[ok]
OP = {'주 모델 (추정 밝기 + 2단계 로지스틱)': oof_pred(main_fp(C_MAIN)),
      '2단계 로지스틱 (사진+해높이+구름)': oof_pred(two_stage(logit(0.01), PLUS)),
      '2단계 규칙 (if 한 줄씩)': oof_pred(two_stage(stump, PLUS)),
      '2단계 규칙 (해 높이만)': oof_pred(two_stage(stump, ['sun_alt'])),
      '규칙4 물리 공식': gr(Rule4Physics().predict(tr), tr.cls.values)[ok]}
gt = gtr[ok]; sct = tr.scene.values[ok]; ugt = np.unique(sct); it = {g: np.where(sct == g)[0] for g in ugt}
rng2 = np.random.default_rng(0); BT = [np.concatenate([it[g] for g in rng2.choice(ugt, len(ugt))]) for _ in range(1000)]
OB = {k: np.array([f1(gt[i], p[i]) for i in BT]) for k, p in OP.items()}
print('학습 쪽 입구', len(gt), '장 ·', len(ugt), '장면')
m = '주 모델 (추정 밝기 + 2단계 로지스틱)'
op = pd.DataFrame([{'A': m, 'B': b, 'A OOF F1': round(f1(gt, OP[m]), 3), 'B OOF F1': round(f1(gt, OP[b]), 3), 'F1 차이': round(float((OB[m] - OB[b]).mean()), 3),
                    '95% 하한': round(float(np.percentile(OB[m] - OB[b], 2.5)), 3), '95% 상한': round(float(np.percentile(OB[m] - OB[b], 97.5)), 3)} for b in OP if b != m])
op['A 우위 (하한>0)'] = op['95% 하한'] > 0
op.to_csv(DATA / '등급2단계_구조교차_학습OOF짝차이.csv', index=False, encoding='utf-8-sig'); display(op)
